# Scaled Dot-Product Attention: Step-by-Step Implementation

This notebook walks through the progressive implementation of the attention mechanism, starting from the simplest possible version and gradually adding complexity to approach a more complete transformer-style attention block.

The levels of complexity are described below:
- basic implementation
- \+ batch support
- \+ multi head attention
- \+ masking
- \+ dropout

In [1]:
import numpy as np

In [255]:
# embedding size of 512, 
# total tokens of 10

d_model = 512
total_size = 10

In [256]:
# Random inputs solely for the forward pass (inputs don't matter since no backward pass - they are just here to exist)
x_input = np.random.randn(total_size, d_model)

## Basic Implementation:
The following implementation provides a **basic, non-batched** version of **scaled dot-product attention**, capturing the essential computation:
- Linear projections for queries ($Q$), keys ($K$), and values ($V$)
- Dot product between $Q$ and $K$
- Scaling by $\sqrt{d_k}$ for numerical stability
- Softmax normalization of attention scores
- Final output as a weighted sum of value vectors

This version is intended to demonstrate the **core mechanics** without additional features like batching, masking, dropout, or multi-head attention. These will be added step-by-step in later sections.


In [257]:
class Attention:
    def __init__(self, d_k):
        self.v = np.random.randn(d_k, d_k)
        self.q = np.random.randn(d_k, d_k)
        self.k = np.random.randn(d_k, d_k)
        self.d_k = d_k

    def softmax(self, x, axis=None):
        x_shifted = x - x.max(axis=axis, keepdims=True) # for numerical stability
        return np.exp(x_shifted) / np.exp(x_shifted).sum(axis=axis, keepdims=True)
    
    def forward(self, X):
        # Linear projections
        Q = X @ self.q
        K = X @ self.k
        V = X @ self.v

        # Scaled dot product
        QK = Q @ K.T
        div_term = self.d_k ** 0.5
        QK_scaled = QK / div_term

        # Attention between tokens
        self.attention = self.softmax(QK_scaled, axis=1)

        # Final projection
        output = self.attention @ V

        return output

In [258]:
attn = Attention(512)

In [259]:
attn.forward(x_input).shape, attn.attention.shape

((10, 512), (10, 10))

## Adding Batch Support:

This section extends the basic attention implementation to support **batched input**.

The input tensor now follows the standard shape:

`(batch_size, sequence_length, d_model)`

Key updates include:

- Supporting batch-wise computation of $Q$, $K$, and $V$
- Computing attention scores for each sequence in the batch simultaneously
- Ensuring softmax is applied correctly across the sequence dimension within each batch
- Maintaining output shape consistency: `(batch_size, sequence_length, d_model)`

This implementation is functionally equivalent to the basic implementation but generalized to operate on multiple sequences in parallel.

## Implementing Multi-Head Attention:

In this section, we enhance the attention mechanism by implementing **multi-head attention**, which allows the model to learn many different patterns *(similar to having multiple filters in a convolutional neural network)*.

Key concepts introduced here:

- Splitting the input projections ($Q$, $K$, and $V$) into multiple heads along the feature dimension.
- Performing scaled dot-product attention independently for each head in parallel.
- Concatenating the outputs of all heads back into a single tensor.
- Applying a final linear projection to combine the multi-head outputs into the original feature dimension.

Multi-head attention increases the model’s ability to capture diverse aspects of the input sequences and has become a standard building block in transformer architectures.

This implementation assumes batched inputs and builds directly on the batching functionality added in the previous section.